# KF Extensions — Fault Detection, Sequential Processing, Steady-State, Prediction & Smoothing

*Course 2 — Linear Kalman Filter, Part 4. Practical add-ons to the [linear KF](08_Deriving_the_Linear_Kalman_Filter.ipynb): rejecting bad measurements, speeding up vector measurements, running at constant gain, and estimating **past** and **future** states.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 Detecting Faulty Measurements

- Sensors glitch. We want to **detect and discard** bad measurements: keep the time update (1a–1c), but skip the measurement update (set $L_k=0$, so $\hat x_k^+=\hat x_k^-$, $\Sigma^+=\Sigma^-$).

- The KF already computes everything we need: predicted measurement $\hat{z}_k = C_k\hat{x}_k^- + D_k u_k$, innovation $\tilde{z}_k = z_k-\hat{z}_k$, and innovation covariance

$$
\Sigma_{\tilde{z},k} = C_k\Sigma_{\tilde{x},k}^{-}C_k^{T} + \Sigma_{\tilde{v}}.
$$

  → We know both *what we expected to measure* and *how uncertain that expectation is*. A measurement that lands far outside that uncertainty is suspect.

- **→ Intuition:** the filter carries its own "was that plausible?" yardstick. No extra model needed — just compare surprise against expected surprise.

### 🧩 Measurement Validation Gating (NEES / $\chi^2$ test)

- Form the **normalized estimation error squared (NEES)** — a scalar "how many sigmas out" score:

$$
e_k^{2} = \tilde{z}_k^{T}\,\Sigma_{\tilde{z},k}^{-1}\,\tilde{z}_k.
$$

  → Weight the innovation by the inverse of its own covariance: a whitened, unit-scale measure of surprise. Small = expected; large = anomalous.

- Under the model, $e_k^2$ has a **chi-squared distribution with $m$ degrees of freedom** ($m = \dim z_k$). Discard the measurement if it exceeds the upper critical value $\chi_U^2(\alpha,m)$:

$$
\text{if } e_k^2 > \chi_U^2(\alpha,m) \Rightarrow \text{reject } (L_k=0); \quad \text{else accept.}
$$

  → Because a whitened sum of squared Gaussians is chi-squared, we can set a principled threshold: e.g. $\alpha=0.01$ discards only the 1% most-extreme *valid* measurements. In Octave: `X2U = chi2inv(1-alpha, m);` (precompute once — it depends only on $m,\alpha$).

- **→ Intuition:** it's a statistical "gate." If too many *consecutive* measurements are rejected, the sensor may truly have failed, or the state estimate got lost — then it helps to **"bump up"** $\Sigma^+ \leftarrow Q\,\Sigma^+$ ($Q>1$) so the filter re-opens to new data.

```octave
zhat = Cd*xhat + Dd*u(:,k);
zerror = z(:,k) - zhat;
SigmaZ = Cd*SigmaX*Cd' + SigmaV;
nees = zerror'/SigmaZ*zerror;
X2U = chi2inv(1-alpha, length(zhat));
if nees <= X2U                       % gate passed -> normal update
  L = SigmaX*Cd'/SigmaZ;
  xhat = xhat + L*zerror;
  SigmaX = SigmaX - L*Cd*SigmaX;
end                                  % else: coast (L=0)
```

### 🧩 Sequential Processing of Vector Measurements

- The costly line in the KF is inverting $\Sigma_{\tilde{z},k}$ in step 2a — an $\mathcal{O}(m^3)$ operation for an $m$-dimensional measurement.

- If sensors are **uncorrelated** ($\Sigma_{\tilde{v}} = \operatorname{diag}(\sigma_{v_1}^2,\dots,\sigma_{v_m}^2)$), process the $m$ scalars **one at a time**. For measurement $i$:

$$
L_{k:i} = \frac{\Sigma_{\tilde{x},k:i-1}^{+}C_{k:i}}{C_{k:i}^{T}\Sigma_{\tilde{x},k:i-1}^{+}C_{k:i} + \sigma_{v_i}^{2}}, \quad
\hat{x}_{k:i}^{+} = \hat{x}_{k:i-1}^{+} + L_{k:i}\big(z_{k:i} - C_{k:i}^{T}\hat{x}_{k:i-1}^{+}\big).
$$

  → Each scalar update replaces the matrix inverse with a **scalar division** ($\mathcal{O}(1)$). Start from $\hat x_{k:0}^+ = \hat x_k^-$ and feed the output of update $i$ into update $i{+}1$; after all $m$, $\hat x_k^+ = \hat x_{k:m}^+$.

- If sensors **are** correlated, first **decorrelate** by the Cholesky factor $\Sigma_{\tilde v}=\mathcal S_v\mathcal S_v^T$: define $\bar z_k = \mathcal S_v^{-1}z_k = \mathcal S_v^{-1}Cx_k + \bar v_k$ with $\operatorname{cov}(\bar v_k)=I$, then process $\bar z_k$'s components sequentially. (LDL decomposition works too, avoiding square roots.)

- **→ Intuition:** turning one big matrix inverse into $m$ tiny divisions is a large speed-up for high-dimensional sensors — the same trick embedded systems use to stay real-time.

### 🧩 Steady-State Kalman Filter

- Plotting $\Sigma_{\tilde{x},k}^{-}$ and $\Sigma_{\tilde{x},k}^{+}$ over time, they usually show a transient then **converge to constants** — and so does $L_k \to L_{ss}$.

  → If the gain settles, why recompute covariances forever? Freeze it.

- Setting $\Sigma_{k+1}^{\pm}=\Sigma_{k}^{\pm}$ in the filter yields the **discrete algebraic Riccati equation (DARE)**:

$$
\Sigma_{\tilde{x},ss}^{-} = \Sigma_{\tilde{w}} + A\Sigma_{\tilde{x},ss}^{-}A^{T} - A\Sigma_{\tilde{x},ss}^{-}C^{T}\big[C\Sigma_{\tilde{x},ss}^{-}C^{T}+\Sigma_{\tilde{v}}\big]^{-1}C\Sigma_{\tilde{x},ss}^{-}A^{T},
$$

  then $L_{ss} = \Sigma_{\tilde{x},ss}^{-}C^{T}[C\Sigma_{\tilde{x},ss}^{-}C^{T}+\Sigma_{\tilde{v}}]^{-1}$.

  → Solve *once, offline* for the fixed-point covariance and the constant gain. The online filter then drops all covariance math:

$$
\hat{x}_k^{+} = A\hat{x}_{k-1}^{+} + Bu_{k-1} + L_{ss}\big(z_k - C(A\hat{x}_{k-1}^{+}+Bu_{k-1}) - Du_k\big).
$$

- In Octave: `[Lss, SigMinus, SigPlus] = dlqe(Ad, eye(nx), Cd, SigmaW, SigmaV);`

- **→ Intuition:** a steady-state filter is **sub-optimal** (constant gain ≠ optimal time-varying gain) but nearly identical after the transient decays, at a huge computational saving. Requires constant $A,B,C,D$ and **detectability + stabilizability** (see [05](05_System_Dynamics_Modes_Observability_Controllability.ipynb)).

### 🧩 Continuous-Time KF (Kalman–Bucy)

- Redo the analysis as $\Delta t\to 0$ to get a **continuous-time** estimator:

$$
\dot{\hat{x}}(t) = A\hat{x}(t) + Bu(t) + L(t)\big[z(t) - C\hat{x}(t) - Du(t)\big],
$$
$$
L(t) = \Sigma_{\tilde{x}}(t)C^{T}S_v^{-1}, \qquad \dot{\Sigma}_{\tilde{x}}(t) = A\Sigma_{\tilde{x}} + \Sigma_{\tilde{x}}A^{T} + B_wS_wB_w^{T} - \Sigma_{\tilde{x}}C^{T}S_v^{-1}C\Sigma_{\tilde{x}}.
$$

  → The covariance now obeys a **differential Riccati equation**: the Lyapunov terms ($A\Sigma+\Sigma A^T + B_wS_wB_w^T$) grow uncertainty from noise, while $-\Sigma C^TS_v^{-1}C\Sigma$ shrinks it from measurements. Same predict/correct tug-of-war, in rate form.

- In steady state ($\dot\Sigma=0$) this becomes the **continuous algebraic Riccati equation** (`lqe.m`). Viewed in the frequency domain, the Kalman–Bucy filter is an **optimal low-pass filter**: gain $L$ sets the corner frequency, balancing state-noise tracking vs. sensor-noise rejection — the ratio $S_w/S_v$ sets convergence speed.

- **→ Intuition:** the KF is, at heart, an *adaptive optimal filter* — it automatically chooses how much to smooth based on the noise statistics, rather than you hand-designing a cutoff.

### 🧩 Kalman-Filter Prediction (estimating the future)

- Three filtering objectives: **filtering** (estimate *now*), **prediction** (estimate the *future*), **smoothing** (estimate the *past*).

- Prediction estimates $x_m$ for $m>k$ using only $\mathbb{Z}_k$:

$$
\hat{x}_{m|k}^{-} = A^{\,m-k}\hat{x}_k^{+} + \sum_{i=k}^{m-1}A^{\,m-i-1}B\,\mathbb{E}[u_i], \qquad
\Sigma_{\tilde{x},m|k}^{-} = A^{\,m-k}\Sigma_{\tilde{x},k}^{+}\big(A^{\,m-k}\big)^{T} + \sum_{j=1}^{m-k}A^{\,j}\Sigma_{\tilde{w}}\big(A^{\,j}\big)^{T}.
$$

  → Just **iterate the prediction step $m-k$ times** with no measurement updates: propagate the current estimate forward and let the covariance grow by accumulated process noise. Often $\mathbb{E}[u_i]=0$, giving $\hat x_{m|k}^- = A^{m-k}\hat x_k^+$.

- **→ Intuition:** prediction bounds **widen** the further ahead you look, because unknown future inputs and noise inject uncertainty you can't yet correct. (Fixed-point, fixed-lead, and fixed-interval prediction are variants of this same extrapolation.)

### 🧩 Kalman-Filter Smoothing (estimating the past)

- Smoothing estimates $x_m$ for $m<N$ using **all** data $\mathbb{Z}_N$ — an offline, post-analysis refinement. **Fixed-interval smoothing** runs a forward KF (saving $\hat x_k^-,\hat x_k^+,\Sigma_k^-,\Sigma_k^+$), then a **backward pass**:

$$
\hat{x}_{m|N}^{+} = \hat{x}_m^{+} + \lambda_m\big(\hat{x}_{m+1|N}^{+} - \hat{x}_{m+1}^{-}\big), \qquad \lambda_m = \Sigma_{\tilde{x},m}^{+}A_m^{T}\big(\Sigma_{\tilde{x},m+1}^{-}\big)^{-1},
$$
$$
\Sigma_{\tilde{x},m|N}^{+} = \Sigma_{\tilde{x},m}^{+} + \lambda_m\big[\Sigma_{\tilde{x},m+1|N}^{+} - \Sigma_{\tilde{x},m+1}^{-}\big]\lambda_m^{T}, \quad m=N-1,\dots,0.
$$

  → Sweep backward from the end. Each past estimate is nudged by how much the *future* smoothed estimate disagreed with what was predicted forward, weighted by the **smoother gain** $\lambda_m$. Start with $\hat x_{N|N}^+ = \hat x_N^+$.

- **→ Intuition:** the smoothed covariance term in brackets is negative-semidefinite, so smoothing is **strictly more certain** than filtering — using future data to sharpen the past. Ideal when you have the whole record and want the best possible reconstruction (offline analysis, lab data).

### 🧩 Summary

- **Fault detection:** gate each measurement with **NEES** $e_k^2 = \tilde z_k^T\Sigma_{\tilde z,k}^{-1}\tilde z_k$ against a **$\chi^2$** threshold; reject outliers by setting $L_k=0$ and coasting.

- **Sequential processing:** for uncorrelated sensors, replace the $\mathcal O(m^3)$ inverse with $m$ scalar updates; decorrelate first (Cholesky/LDL) if needed.

- **Steady-state KF:** solve the **DARE** once for constant $L_{ss}$ (`dlqe`), dropping all online covariance math — nearly optimal after the transient. Its continuous cousin is the **Kalman–Bucy** filter (differential Riccati / optimal low-pass).

- **Prediction:** iterate the prediction step to reach future $x_{m|k}$; bounds widen with horizon.

- **Smoothing:** a forward KF + backward pass yields $\hat x_{m|N}^+$ using all data — strictly tighter than filtering.

---
*Next: [11 · Target Tracking — Models, Polar Conversion, IMM & α-β-γ Filters](11_Target_Tracking_IMM_AlphaBetaGamma.ipynb).*